# AI_DoRA: Full-Scale Parameter-Efficient Dynamic Rank Adaptation Experiments

This notebook executes full-scale fine-tuning experiments for the **AI_DoRA** framework on Google Colab using a Tesla T4 GPU (16 GB VRAM).

### Experimental Foundations & Operational Policies
1. **Exploratory Default (30 Epochs)**: Standard fine-tuning duration for new model architectures (`roberta-base`, `distilroberta-base`, `bert-base-uncased`), datasets (`sst2`, `imdb`, `ag_news`), and algorithmic ablations. Provides deep convergence while keeping execution time manageable on standard Colab runtimes.
2. **Baseline Preservation**: The verified 60-epoch baseline checkpoint (`roberta_sst2_latest.pt` with 99.74% train accuracy and 93.69% validation accuracy) is stored in `/content/drive/MyDrive/AI_DoRA_checkpoints/` and is strictly isolated. All new runs write to `/content/drive/MyDrive/AI_DoRA_experiments/`.
3. **Persistent Google Drive Storage**: Checkpoints, epoch evaluation histories, summary JSON files, and comparison plots are saved directly to Google Drive, ensuring complete recovery across sessions.
4. **Resumable Atomic Checkpointing**: Two checkpoints are maintained per run: `*_latest.pt` (full resumable training state with optimizer, RNG, and pruner internals) and `*_best.pt` (weights achieving the highest validation accuracy).

---
### Cell 1: Check Available GPU and Runtime Environment

In [ ]:
# 1. Check GPU and CUDA runtime environment
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU is not enabled! Please navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

### Cell 2: Mount Google Drive & Setup Persistent Storage

> **Safety Notice**: All new experimental artifacts are directed to `/content/drive/MyDrive/AI_DoRA_experiments/`. The directory is structured into `checkpoints/`, `results/`, `logs/`, and `plots/`. The historical 60-epoch baseline directory (`/content/drive/MyDrive/AI_DoRA_checkpoints/`) is left untouched.

In [ ]:
# 2. Mount Google Drive and initialize experiment directories
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE_DRIVE_DIR = Path('/content/drive/MyDrive/AI_DoRA_experiments')
CHECKPOINT_DIR = BASE_DRIVE_DIR / 'checkpoints'
RESULTS_DIR = BASE_DRIVE_DIR / 'results'
LOGS_DIR = BASE_DRIVE_DIR / 'logs'
PLOTS_DIR = BASE_DRIVE_DIR / 'plots'

for directory in [CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Persistent experiment directories verified in Google Drive:")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Results:     {RESULTS_DIR}")
print(f"  Logs:        {LOGS_DIR}")
print(f"  Plots:       {PLOTS_DIR}")

### Cell 3: Clone or Update AI_DoRA Repository

Clones or pulls the `extensions` branch, which contains the 30-epoch configurations, checkpoint resilience, and automated reporting scripts.

In [ ]:
# 3. Clone or pull repository branch
import os

REPO_URL = "https://github.com/Bhavya12381/AI_DoRA.git"
TARGET_DIR = "/content/AI_DoRA"
BRANCH = "extensions"

if not os.path.exists(TARGET_DIR):
    !git clone -b {BRANCH} {REPO_URL} {TARGET_DIR}
else:
    print(f"{TARGET_DIR} already present. Fetching latest commits...")
    %cd {TARGET_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {TARGET_DIR}
!git status

### Cell 4: Install Dependencies

In [ ]:
# 4. Install project dependencies
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install datasets transformers pandas scikit-learn matplotlib

### Cell 5: Dependency and CUDA Verification

Validates that PyTorch detects the Tesla T4 GPU, required ML libraries are loaded, and internal AI_DoRA modules (`models`, `pruner`, `checkpoint`, `compact`, `benchmark`) can be imported without errors.

In [ ]:
# 5. Verify dependencies and internal DoRA imports
import sys
if "/content/AI_DoRA" not in sys.path:
    sys.path.insert(0, "/content/AI_DoRA")

import torch
import transformers
import datasets
import pandas as pd
import sklearn
import matplotlib

print("--- Environment & Dependency Verification ---")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")

# Verify AI_DoRA core modules
from src.dora.models import setup_adapted_model, detect_model_family
from src.dora.pruner import DynamicRankPruner
from src.dora.regularization import dem_loss
from src.dora.checkpoint import save_checkpoint, load_checkpoint
from src.dora.compact import export_compact_model, verify_compact_equivalence
from src.dora.benchmark import compare_inference_variants
print("All AI_DoRA modules verified successfully!")

### Cell 6: Run Fast End-to-End Smoke Test

Executes a rapid 1-epoch subset smoke test verifying the complete pipeline:
- Dataset streaming and tokenization
- DoRA parameter injection across linear projections
- Gradient computation and backpropagation
- Dynamic rank pruning step (EMA importance tracking and masking)
- Validation evaluation (accuracy and F1 score computation)
- Atomic checkpoint generation
- Compact physical adapter parameter export
- Multi-batch inference latency and memory benchmarking

Expected duration: < 30 seconds.

In [ ]:
# 6. Execute smoke test
!python -u experiments/scripts/run_experiment.py \
    --model_name roberta-base \
    --dataset_name sst2 \
    --smoke_test \
    --export_compact \
    --benchmark_inference

### Cell 7: Execute Primary Full-Scale Experiment: RoBERTa-base on SST-2 (30 Epochs)

This is the first full-scale training run using the updated 30-epoch experimental protocol.

| Specification | Value |
| :--- | :--- |
| Base Model | `roberta-base` (12 layers, 72 adaptive projections) |
| Dataset | Stanford Sentiment Treebank (SST-2, 67,349 train / 872 val) |
| Training Duration | 30 Epochs (31,590 total steps at batch size 64) |
| Learning Rate | 8e-4 with AdamW optimizer |
| Pruning Schedule | Cubic rank budget decay from 216 to 144 between step 4,738 (15%) and step 15,795 (50%) |
| DEM Regularization | $\gamma = 0.5$ |
| Estimated Tesla T4 Runtime | ~1.5 - 1.65 hours (~3.0 - 3.3 min/epoch) |

Checkpoints (`roberta_sst2_latest.pt` and `roberta_sst2_best.pt`), epoch validation metrics (`val_metrics.csv`), and execution logs are continuously saved directly to Google Drive.

In [ ]:
# 7. Run full-scale 30-epoch RoBERTa-base SST-2 experiment
!python -u experiments/scripts/run_experiment.py \
    --config experiments/configs/models/model_roberta_base.json \
    --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
    --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
    --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs \
    --export_compact \
    --benchmark_inference

### Cell 8: Verify Model Checkpoints and Results Saved to Google Drive

Inspects the Google Drive folder to verify that:
1. `roberta_sst2_latest.pt` (full resumable training state) is saved.
2. `roberta_sst2_best.pt` (state with highest validation accuracy) is saved.
3. `compact_adapter.pt` (pruned rank-2 physical adapter) and configuration JSONs are saved.
4. `val_metrics.csv` logs epoch-by-epoch validation accuracy and F1 scores.
5. Result summary JSON is generated.

In [ ]:
# 8. Verify saved experimental assets in Google Drive
import json
from pathlib import Path
import pandas as pd

checkpoints_root = Path('/content/drive/MyDrive/AI_DoRA_experiments/checkpoints')
results_root = Path('/content/drive/MyDrive/AI_DoRA_experiments/results')
logs_root = Path('/content/drive/MyDrive/AI_DoRA_experiments/logs')

print("=== Checkpoints in Google Drive ===")
for p in sorted(checkpoints_root.rglob("*.pt")):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  {p.relative_to(checkpoints_root)} ({size_mb:.2f} MB)")

print("\n=== Results in Google Drive ===")
for p in sorted(results_root.glob("*.json")):
    print(f"  {p.name}")

print("\n=== Logs in Google Drive ===")
for p in sorted(logs_root.glob("*.log")):
    print(f"  {p.name}")

# Display validation history table
val_csvs = list(checkpoints_root.rglob("val_metrics.csv"))
if val_csvs:
    print(f"\n=== Validation Metrics Progression ({val_csvs[0].relative_to(checkpoints_root)}) ===")
    df_val = pd.read_csv(val_csvs[0])
    print(df_val.to_string(index=False))

### Cell 9: Resume an Interrupted Run

If Colab experiences a runtime timeout or disconnect during training, run this cell.
Passing `--resume_from_checkpoint` automatically:
- Detects the completed epoch and resumes at `epoch + 1`
- Restores the exact model adapter weights and classification head
- Restores the AdamW optimizer state and step counters
- Restores the dynamic pruner EMA importance scores and pruning mask
- Restores the Torch and CUDA RNG seeds
- Preserves and appends to the existing `val_metrics.csv`

In [ ]:
# 9. Resume an interrupted experiment from latest checkpoint
import glob
from pathlib import Path

# Find latest checkpoint saved in Google Drive
candidates = glob.glob('/content/drive/MyDrive/AI_DoRA_experiments/checkpoints/**/roberta_sst2_latest.pt', recursive=True)

if candidates:
    latest_checkpoint = candidates[0]
    print(f"Found checkpoint to resume: {latest_checkpoint}")
    !python -u experiments/scripts/run_experiment.py \
        --resume_from_checkpoint "{latest_checkpoint}" \
        --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
        --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs \
        --export_compact \
        --benchmark_inference
else:
    print("No latest checkpoint found. Specify the full path to resume a specific experiment.")

### Cell 10: Execute Additional Model, Dataset, and Ablation Experiments

Execute remaining experimental variants on Google Colab:

**Hardware & Runtime Estimates (Tesla T4):**
- `distilroberta-base` on SST-2 (30 epochs): ~48–54 minutes
- `bert-base-uncased` on SST-2 (30 epochs): ~1.4–1.5 hours
- `roberta-base` on IMDb (20 epochs): ~1.2–1.4 hours
- `roberta-base` on AG News (20 epochs): ~1.5–1.8 hours
- Ablations (Linear Schedule, No EMA, No DEM): ~1.5 hours each

In [ ]:
# 10. Additional Experiments (Uncomment the desired block to execute)

# --- 1. DistilRoBERTa-base on SST-2 (30 epochs, ~50 min) ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/models/model_distilroberta_base.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs \
#     --export_compact \
#     --benchmark_inference

# --- 2. BERT-base-uncased on SST-2 (30 epochs, ~1.4 hrs) ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/models/model_bert_base.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs \
#     --export_compact \
#     --benchmark_inference

# --- 3. Multi-Dataset: IMDb Sentiment Classification (20 epochs) ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/datasets/dataset_imdb.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs

# --- 4. Multi-Dataset: AG News Topic Classification (20 epochs) ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/datasets/dataset_agnews.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs

# --- 5. Ablations: Linear Rank Budget Schedule ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/ablations/schedule_linear.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs

# --- 6. Ablations: Instantaneous Importance (No EMA) ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/ablations/smoothing_no_ema.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs

# --- 7. Ablations: No DEM Regularization ---
# !python experiments/scripts/run_experiment.py \
#     --config experiments/configs/ablations/regularization_no_dem.json \
#     --checkpoint_dir /content/drive/MyDrive/AI_DoRA_experiments/checkpoints \
#     --output_dir /content/drive/MyDrive/AI_DoRA_experiments/results \
#     --log_dir /content/drive/MyDrive/AI_DoRA_experiments/logs

print("Select and uncomment any desired experiment block above to execute.")

### Cell 11: Aggregate Results and Generate Comparison Plots

Aggregates all JSON results across runs into unified CSV and JSON comparison tables, produces publication-ready visualization charts, and displays them inline in the notebook.

In [ ]:
# 11. Aggregate results and plot comparisons
from pathlib import Path
from IPython.display import Image, display

RESULTS_DIR = "/content/drive/MyDrive/AI_DoRA_experiments/results"
PLOTS_DIR = "/content/drive/MyDrive/AI_DoRA_experiments/plots"

# 1. Aggregate JSON outputs into summary CSVs and JSONs
!python experiments/scripts/aggregate_results.py --results_dir {RESULTS_DIR}

# 2. Generate comparison charts
!python experiments/scripts/plot_results.py --results_dir {RESULTS_DIR} --plots_dir {PLOTS_DIR}

# 3. Display generated plots inline
plots_path = Path(PLOTS_DIR)
plot_images = sorted(plots_path.glob("*.png"))
if plot_images:
    for img in plot_images:
        print(f"\n--- Displaying: {img.name} ---")
        display(Image(filename=str(img)))
else:
    print("No plot images found yet in plots directory.")